# 23 Reading Parquet

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand what makes Parquet fast (columnar layout, embedded schema, row-group statistics, compression), read and write it, see column pruning and predicate pushdown in the query plan, and handle schema differences between files with <code>mergeSchema</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Parquet is the storage format underneath Delta Lake, so understanding it explains most of the lakehouse's performance. Interviewers ask "why is Parquet better than CSV for analytics?" all the time, and the answer (columnar storage + statistics + compression) also explains column pruning, predicate pushdown and data skipping.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How a Parquet file is organized</b><br>
<code>`</code><br>file.parquet<br>├── Row group 1 (e.g. ~128 MB of rows)<br>│     ├── column chunk: order_id   (compressed, encoded, with min/max stats)<br>│     ├── column chunk: amount     (min=3.5, max=980.0)<br>│     └── column chunk: country    (dictionary encoded: IN, UK, AE)<br>├── Row group 2 ...<br>└── Footer: schema + row group metadata + column statistics<br><code>`</code>
<ul><li><b>Columnar:</b> the values of one column are stored together, so reading 3 of 50 columns reads only those 3 (<b>column pruning</b>)</li><li><b>Self-describing:</b> the schema and types are in the footer, so no inference is needed</li><li><b>Statistics:</b> min/max per column per row group let readers skip row groups that can't match a filter (<b>predicate pushdown</b>)</li><li><b>Compression and encoding:</b> similar values stored together compress very well (snappy by default, dictionary and run-length encoding)</li><li><b>Immutable files:</b> you don't update a Parquet file, you write new ones. Delta Lake adds a transaction log on top to provide updates, deletes and ACID</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A telco keeps 2 years of call records: 40 billion rows, 60 columns. As CSV it is 18 TB, and every query reads all of it. As Parquet it is about 2.5 TB, a query on <code>call_date</code>, <code>cell_id</code> and <code>duration</code> reads only those 3 columns, and a filter on one week skips the row groups whose min/max dates are outside that week. A dashboard query goes from hours to seconds.
</div>

| | CSV | JSON | Parquet | Delta |
|---|---|---|---|---|
| Layout | Row, text | Row, text | Columnar, binary | Parquet + transaction log |
| Schema | None (inferred) | Inferred | Embedded | Embedded + enforced |
| Compression | Poor | Poor | Excellent | Excellent |
| Column pruning | No | No | Yes | Yes |
| Skip data with stats | No | No | Row groups | Files + row groups |
| Updates / ACID | No | No | No | Yes |

## Syntax

```python
df.write.mode("overwrite").parquet(path)                       # write
df = spark.read.parquet(path)                                   # read (schema from the files)
df = spark.read.option("mergeSchema", True).parquet(path)       # combine differing schemas
df.write.option("compression", "zstd").parquet(path)            # choose codec
spark.sql(f"SELECT * FROM parquet.`{path}`")                    # query files directly in SQL
```

## Setup: volume and a sample dataset

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the lesson folder, generates 1 million sales rows, and writes them as both CSV and Parquet.<br><br>
<b>Why it matters</b><br>With the same data in two formats you can compare size, schema handling and query plans side by side.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>BASE</code> path, then confirmation that both formats were written.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson23"
print("BASE =", BASE)

In [0]:
from pyspark.sql import functions as F

sales = (
    spark.range(1_000_000)
    .withColumnRenamed("id", "sale_id")
    .withColumns({
        "sale_date": F.expr("date_add(date'2024-01-01', cast(sale_id % 366 AS INT))"),
        "country": F.element_at(F.array(*[F.lit(c) for c in ["IN", "UK", "AE", "US"]]), (F.col("sale_id") % 4 + 1).cast("int")),
        "store_id": (F.col("sale_id") % 200).cast("int"),
        "amount": F.round((F.col("sale_id") * 7919 % 100000) / 100.0, 2),
        "note": F.lit("standard sale with a fairly repetitive description text"),
    })
)

sales.write.mode("overwrite").option("header", True).csv(f"{BASE}/sales_csv")
sales.write.mode("overwrite").parquet(f"{BASE}/sales_parquet")
print("written")

## 1. Look at the files and compare sizes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the files Spark wrote and adds up the size of each format.<br><br>
<b>Why it matters</b><br>A Parquet "file" written by Spark is really a <b>folder</b> of part files, one per task, plus marker files such as <code>_SUCCESS</code>. And because similar values are stored together and compressed, Parquet is usually several times smaller than CSV.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Several <code>part-...snappy.parquet</code> files, then sizes in MB. Parquet is a fraction of the CSV size (often 4 to 10 times smaller).
</div>

In [0]:
import os

def folder_size_mb(path):
    total = 0
    for root, _, names in os.walk(path):
        total += sum(os.path.getsize(os.path.join(root, n)) for n in names)
    return round(total / 1024 / 1024, 2)

print([n for n in sorted(os.listdir(f"{BASE}/sales_parquet")) if not n.startswith(".")])
csv_mb, pq_mb = folder_size_mb(f"{BASE}/sales_csv"), folder_size_mb(f"{BASE}/sales_parquet")
print(f"CSV: {csv_mb} MB | Parquet: {pq_mb} MB | ratio: {csv_mb / pq_mb:.1f}x smaller")

## 2. The schema travels with the data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads both formats back without any options and compares the schemas.<br><br>
<b>Why it matters</b><br>Parquet stores names <b>and types</b> in the file footer, so you get <code>date</code>, <code>int</code> and <code>double</code> back for free. CSV comes back as strings unless you supply a schema or pay for inference.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The CSV schema has every column as <code>string</code>. The Parquet schema has <code>bigint</code>, <code>date</code>, <code>string</code>, <code>int</code>, <code>double</code>, <code>string</code>.
</div>

In [0]:
spark.read.option("header", True).csv(f"{BASE}/sales_csv").printSchema()
spark.read.parquet(f"{BASE}/sales_parquet").printSchema()

## 3. Column pruning and predicate pushdown in the plan

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads Parquet, selects 2 columns and filters on one, then shows the physical plan.<br><br>
<b>Why it matters</b><br>This is the proof of Parquet's speed. <code>ReadSchema</code> lists only the columns the query needs, and <code>PushedFilters</code> shows the filter handed to the Parquet reader, which uses row-group min/max statistics to skip data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>FileScan parquet</code> node with <code>ReadSchema: struct&lt;sale_date:date,amount:double&gt;</code> (only 2 of 6 columns) and <code>PushedFilters: [IsNotNull(amount), GreaterThan(amount,900.0)]</code>.
</div>

In [0]:
pq = spark.read.parquet(f"{BASE}/sales_parquet")
query = pq.filter(F.col("amount") > 900).select("sale_date", "amount")
query.explain()
print("rows:", query.count())

## 4. Look inside a Parquet file: row groups and statistics

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the <code>pyarrow</code> library (pre-installed on Databricks) to read the footer of one part file.<br><br>
<b>Why it matters</b><br>Seeing the row groups and min/max statistics makes "predicate pushdown" concrete: a filter like <code>sale_date = '2024-12-31'</code> can skip any row group whose <code>sale_date</code> max is earlier.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The number of rows and row groups in the file, then the min/max statistics of <code>sale_date</code> and <code>amount</code>. Notice the ranges: <code>sale_date</code> spans the whole year and <code>amount</code> spans 0 to 999.99. Because the data was generated in <code>sale_id</code> order, every file contains every date and amount, so a filter on one week could <b>not</b> skip this row group. See "Why skipping depends on data order" below.
</div>

In [0]:
import pyarrow.parquet as pq_lib

first_file = sorted(n for n in os.listdir(f"{BASE}/sales_parquet") if n.endswith(".parquet"))[0]
meta = pq_lib.ParquetFile(f"{BASE}/sales_parquet/{first_file}").metadata

print("file:", first_file)
print("rows:", meta.num_rows, "| row groups:", meta.num_row_groups, "| columns:", meta.num_columns)
rg = meta.row_group(0)
for i in range(rg.num_columns):
    col = rg.column(i)
    if col.path_in_schema in ("sale_date", "amount"):
        print(f"{col.path_in_schema:<10} min={col.statistics.min}  max={col.statistics.max}  compression={col.compression}")

## 5. Schema evolution across files: `mergeSchema`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a second batch that has an extra column, then reads both folders with and without <code>mergeSchema</code>.<br><br>
<b>Why it matters</b><br>Sources add columns over time. By default Spark takes the schema from one file and <b>silently ignores</b> extra columns in others. <code>mergeSchema=True</code> reads all footers and unions the columns. It is off by default because reading every footer is expensive. Delta Lake handles this more safely (lesson in <code>03_delta_lake</code>).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Without <code>mergeSchema</code>: <code>sale_id</code>, <code>country</code>, <code>amount</code>, <code>batch</code>, with <code>channel</code> missing. With <code>mergeSchema</code>: <code>channel</code> is included, null for the old row and <code>web</code> for the new one.<br><br>
Where did <code>batch</code> come from? The folders are named <code>batch=1</code> and <code>batch=2</code>. Spark treats <code>name=value</code> folders as <b>partitions</b> and turns them into a column. This is called partition discovery, and lesson 25 covers it.
</div>

In [0]:
batch1 = spark.createDataFrame([(1, "IN", 10.0)], "sale_id INT, country STRING, amount DOUBLE")
batch2 = spark.createDataFrame([(2, "UK", 20.0, "web")], "sale_id INT, country STRING, amount DOUBLE, channel STRING")

batch1.write.mode("overwrite").parquet(f"{BASE}/evolving/batch=1")
batch2.write.mode("overwrite").parquet(f"{BASE}/evolving/batch=2")

print("Without mergeSchema:", spark.read.parquet(f"{BASE}/evolving").columns)

merged = spark.read.option("mergeSchema", True).parquet(f"{BASE}/evolving")
print("With mergeSchema:   ", merged.columns)
merged.orderBy("sale_id").show()

## 6. Compression codecs

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same data with <code>snappy</code> (default), <code>zstd</code> and <code>none</code>, and compares the sizes.<br><br>
<b>Why it matters</b><br><code>snappy</code> is fast with good compression. <code>zstd</code> usually compresses better at slightly higher CPU cost, and Databricks often uses it by default for Delta. Uncompressed files are rarely a good idea.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three sizes. <code>none</code> is the largest, and <code>zstd</code> is usually the smallest.
</div>

In [0]:
small = sales.limit(200_000)
for codec in ["snappy", "zstd", "none"]:
    path = f"{BASE}/codec_{codec}"
    small.write.mode("overwrite").option("compression", codec).parquet(path)
    print(f"{codec:<7} {folder_size_mb(path)} MB")

## 7. Query Parquet with SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the Parquet folder directly in SQL with the <code>parquet.</code> prefix, without creating a table.<br><br>
<b>Why it matters</b><br>Handy for quick exploration of raw files. For anything shared or repeated, register a table instead (lesson 24).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue per country, 4 rows.
</div>

In [0]:
spark.sql(f"""
    SELECT country, CAST(sum(amount) AS DECIMAL(18,2)) AS revenue, count(*) AS sales
    FROM parquet.`{BASE}/sales_parquet`
    GROUP BY country
    ORDER BY country
""").show()

## Under the hood

```
SELECT sale_date, amount FROM sales WHERE amount > 900

1. Driver lists files and reads footers (schema)
2. Files are split into tasks (by size, ~128 MB; a row group is never split)
3. Each task, for each row group:
     check footer stats: max(amount) <= 900 ?  -> SKIP the whole row group
     otherwise read ONLY the sale_date and amount column chunks
     decode + decompress, apply the filter row by row
4. Vectorized reader returns batches of rows (columnar batches) to the query
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> reading Parquet is lazy. Only the footer (schema) is touched at read time.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> scanning and filtering need no shuffle.

**Why skipping depends on data order:** min/max statistics only help when values are clustered. If every row group contains amounts from 0 to 1000, no row group can be skipped. Sorting or clustering data by commonly filtered columns before writing (or Delta's `OPTIMIZE ZORDER BY` / liquid clustering) makes skipping effective. You will see this again in `03_delta_lake` and `05_optimization`.

In [0]:
# Optional cleanup: remove this lesson's files.
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: a new column is missing when reading a folder</b><br>
Spark used one file's schema. Use <code>mergeSchema=True</code>, or better, Delta Lake with schema evolution.<br><br>
<b>⛔ Problem: filters don't speed things up</b><br>The filtered values are spread across all row groups, so statistics can't skip anything. Sort or cluster data by the filter column when writing.<br><br>
<b>⛔ Problem: thousands of tiny Parquet files</b><br>Each write task creates a file. Many small files mean lots of footer reads and slow listing. Repartition or coalesce before writing (lesson 24), or use Delta <code>OPTIMIZE</code>.<br><br>
**⛔ Problem: <code>Parquet column cannot be converted</code> errors**<br>Files have conflicting types for the same column (for example <code>int</code> in one, <code>string</code> in another). Rewrite the bad files with a consistent schema.<br><br>
<b>⛔ Problem: trying to update rows in a Parquet file</b><br>Parquet files are immutable. Use Delta Lake for <code>UPDATE</code>, <code>DELETE</code> and <code>MERGE</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Why is Parquet better than CSV for analytics?</b><br>
It is columnar, so queries read only the columns they need. It stores the schema and types, so there's no inference. It keeps min/max statistics per row group, so filters can skip data. And it compresses much better because similar values are stored together.<br><br>

<b>🎤 2. What is column pruning?</b><br>
The optimizer passes the list of required columns down to the reader. With columnar formats like Parquet, only those column chunks are read from storage.<br><br>

<b>🎤 3. What is predicate pushdown in Parquet?</b><br>
Filters are passed to the Parquet reader, which compares them with each row group's min/max statistics and skips row groups that can't contain matching rows.<br><br>

<b>🎤 4. What is a row group?</b><br>
A horizontal slice of a Parquet file, holding one column chunk per column plus statistics. It is the unit of skipping and the smallest unit a reader task works on.<br><br>

<b>🎤 5. What does <code>mergeSchema</code> do, and why is it off by default?</b><br>
It reads the schemas of all files and combines them, so columns that exist only in some files are included. It is off by default because reading every footer is expensive on large datasets.<br><br>

<b>🎤 6. How does Delta Lake relate to Parquet?</b><br>
Delta stores data as Parquet files and adds a transaction log (<code>_delta_log</code>) that records which files make up each table version. That gives ACID transactions, updates and deletes, schema enforcement, time travel and file-level statistics for data skipping.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A query selects 3 columns from a 100-column table stored in Parquet. Which feature lets Spark read only those 3 columns from storage?</b><br>
A. Partition pruning<br>
B. Column pruning on a columnar format<br>
C. Broadcast join<br>
D. Caching<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. What is the underlying file format of data files in a Delta Lake table?</b><br>
A. Avro<br>
B. ORC<br>
C. Parquet<br>
D. JSON<br>
<details><summary><b>Show answer</b></summary><b>C.</b> Delta stores data in Parquet files. The transaction log is stored as JSON (plus Parquet checkpoint) files.</details><br><br>

<b>Q3. Which SQL statement queries Parquet files in a volume directly without creating a table?</b><br>
A. <code>SELECT * FROM '/Volumes/main/raw/sales'</code><br>
B. <code>SELECT * FROM parquet.`/Volumes/main/raw/sales`</code><br>
C. <code>SELECT * FROM FILE('/Volumes/main/raw/sales')</code><br>
D. <code>READ PARQUET '/Volumes/main/raw/sales'</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> The <code>format.`path`</code> syntax queries files directly. <code>read_files(...)</code> is another option.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>From <code>sales</code>, write only the <code>UK</code> rows to <code>BASE/practice/uk_sales</code> as Parquet with <code>zstd</code> compression</li><li>Read it back and prove (with <code>explain</code>) that a query on <code>amount</code> and <code>store_id</code> reads only those columns</li><li>Write a second batch with an extra column <code>promo_code</code> into a sibling folder and read both with <code>mergeSchema</code></li><li>In a comment, explain why sorting by <code>sale_date</code> before writing would help a query filtering one week of data</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Filter and write with zstd
sales.filter(F.col("country") == "UK").write.mode("overwrite").option("compression", "zstd").parquet(f"{BASE}/practice/uk_sales/part=1")

# 2. Column pruning in the plan (look at ReadSchema)
uk = spark.read.parquet(f"{BASE}/practice/uk_sales")
uk.select("store_id", "amount").filter("amount > 500").explain()

# 3. Second batch with an extra column, then merge
extra = spark.createDataFrame([(9_999_999, "UK", 1, 12.5, "SPRING")], "sale_id BIGINT, country STRING, store_id INT, amount DOUBLE, promo_code STRING")
extra.write.mode("overwrite").parquet(f"{BASE}/practice/uk_sales/part=2")
spark.read.option("mergeSchema", True).parquet(f"{BASE}/practice/uk_sales").filter("promo_code IS NOT NULL").show()

# 4. Sorted data puts each week into few row groups with narrow min/max ranges,
#    so a one-week filter can skip almost every other row group.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Parquet is columnar, typed, compressed and self-describing</li><li>Column pruning reads only needed columns. Predicate pushdown skips row groups using min/max stats</li><li>Skipping works best when data is sorted or clustered by the filter columns</li><li>A Spark "Parquet file" is a folder of part files, one per write task</li><li><code>mergeSchema</code> combines schemas across files, but is expensive and off by default</li><li>Parquet files are immutable. Delta Lake = Parquet + transaction log (ACID, updates, time travel)</li><li>Query files directly with <code>parquet.`path`</code> or <code>read_files</code></li></ul>
</div>

| Task | Code |
|---|---|
| Write | `df.write.mode("overwrite").parquet(path)` |
| Read | `spark.read.parquet(path)` |
| Merge schemas | `.option("mergeSchema", True)` |
| Codec | `.option("compression", "zstd")` |
| SQL on files | ``SELECT * FROM parquet.`/Volumes/...` `` |
| Check pruning | `df.select(...).filter(...).explain()` → `ReadSchema`, `PushedFilters` |

## Git commit

```
git add 01_spark_basics/23_reading_parquet.ipynb
git commit -m "add 23 reading parquet notebook"
```